# Select Umbrella Windows

This Jupyter Notebook provides a detailed guide and script for extracting windows for umbrella sampling from a steered molecular dynamics (SMD) simulation. 

The process involves selecting conformations at regular intervals along a collective variable (CV) to use as starting points for umbrella sampling simulations. 

Additionally, it includes a step to discard windows where the molecule is too far from the initial pulling site, as calculating the free energy profile (FEP) at these windows would be pointless.

## Loading and Sorting Data

The data is loaded and sorted by the CV position to ensure a smooth progression along the reaction coordinate.

In [ ]:
import numpy as np
import pandas as pd

def find_nearest(array, value):
    """Find the index and value of the nearest element in an array to a given value."""
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return int(idx), float(array[idx])

# Define the window size in nanometers (1 Å = 0.1 nm)
WINDOW = 0.1

pullx_file = '../1_smd/smd_pullx.xvg'

In [ ]:
# Load and sort data
dat = np.loadtxt(pullx_file, comments=['@', '#'], unpack=True).T[::10, :]
dat = pd.DataFrame(dat, columns=['frame', 'pos'])
dat = dat.sort_values('pos').reset_index(drop=True)

## Windows Extration

* **Define CV steps**

The CV positions are generated at regular intervals (`WINDOW` size) from the initial to the final position.

In [46]:
# Create CV steps
cv_init_pos = dat.iloc[0, -1]
cv_final_pos = dat.iloc[-1, -1]
cv_pos = np.arange(cv_init_pos, cv_final_pos + WINDOW, WINDOW)

* **Select Windows**

For each CV position, find the nearest frame in the SMD data.

In [48]:
# Select windows
windows = np.array([list(find_nearest(dat['pos'], p)) for p in cv_pos])
windows_frames, windows_pos = windows[:, 0], windows[:, 1]

Eventually, check whether you can discard some windows. For example, if the molecule is too far from the initial pulling site, there may be no interaction anymore, and calculating the FEP at these windows would be pointless.

Suppose you want to discard windows where the CV position is beyond a certain threshold (`max_relevant_pos`). You can filter the windows as follows:

In [ ]:
# Define the maximum relevant position (e.g., 5 nm)
max_relevant_pos = 5.0

# Filter windows
relevant_windows = windows[windows_pos <= max_relevant_pos]
relevant_windows_frames, relevant_windows_pos = relevant_windows[:, 0], relevant_windows[:, 1]

## Script generation

A bash script is generated to extract the conformations corresponding to the selected frames using GROMACS' trjconv tool.

The script will generate `.gro` files for each selected frame, which can be used as starting points for umbrella sampling simulations.

In [ ]:
# Write script to extract the conformations chosen as windows
with open("extract_umbrella_windows.sh", "w") as fout:
    fout.write("#!/bin/bash\n")  # Fixed shebang line

    for f in relevant_windows_frames:  # Use relevant_windows_frames instead of windows_frames
        frame = int(f)
        fout.write(f"echo '0' | gmx_mpi trjconv -s ../1_smd/smd.tpr -f ../1_smd/smd.xtc -o conf{frame}.gro -b {frame} -e {frame}\n")